In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping

torch.set_float32_matmul_precision('medium')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Carregar dados
df_series = pd.read_parquet('/home/valentim/divea/data/processed/series_semanais.parquet')
print(f"Series carregadas: {df_series.shape}")

/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


RuntimeError: No CUDA GPUs are available

In [ ]:
def criar_dataset_tft(serie, janela=16, split=0.7):
    df = serie.reset_index()
    df.columns = ['data', 'casos']
    df['time_idx'] = range(len(df))
    df['grupo'] = 'parana'
    df['casos'] = df['casos'].astype(float)
    df['semana'] = df['data'].dt.isocalendar().week.astype(str)
    df['mes'] = df['data'].dt.month.astype(str)
    
    SPLIT = int(len(df) * split)
    
    treino = TimeSeriesDataSet(
        df[df['time_idx'] <= SPLIT],
        time_idx='time_idx', target='casos', group_ids=['grupo'],
        min_encoder_length=janela, max_encoder_length=janela,
        min_prediction_length=4, max_prediction_length=4,
        time_varying_known_categoricals=['semana', 'mes'],
        time_varying_unknown_reals=['casos'],
        target_normalizer=GroupNormalizer(groups=['grupo'], transformation='softplus'),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
    )
    validacao = TimeSeriesDataSet.from_dataset(treino, df, predict=False, stop_randomization=True)
    train_loader = treino.to_dataloader(train=True, batch_size=32, num_workers=0)
    val_loader = validacao.to_dataloader(train=False, batch_size=32, num_workers=0)
    return treino, train_loader, val_loader, df

def treinar_tft(treino, train_loader, val_loader, epochs=80):
    model = TemporalFusionTransformer.from_dataset(
        treino, learning_rate=0.03, hidden_size=32,
        attention_head_size=2, dropout=0.1, hidden_continuous_size=16,
        loss=QuantileLoss(), optimizer='adam', log_interval=-1,
        reduce_on_plateau_patience=4,
    )
    early_stop = EarlyStopping(monitor='val_loss', patience=10, mode='min')
    trainer = Trainer(
        max_epochs=epochs, accelerator='gpu', devices=1,
        callbacks=[early_stop], enable_progress_bar=False,
        log_every_n_steps=1, logger=False,
    )
    trainer.fit(model, train_loader, val_loader)
    return model, trainer

def avaliar_tft(model, val_loader):
    model.eval()
    all_preds, all_reals = [], []
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch
            out = model(x)
            all_preds.append(out['prediction'].cpu().numpy())
            all_reals.append(y[0].cpu().numpy())
    y_pred = np.concatenate(all_preds, axis=0)
    y_real = np.concatenate(all_reals, axis=0)
    for s in range(4):
        mae = mean_absolute_error(y_real[:, s], y_pred[:, s, 3])
        print(f"  Semana +{s+1} MAE: {mae:.1f}")
    print(f"  Media real: {y_real.mean():.1f}")
    return y_pred, y_real

print("Funcoes definidas.")

In [ ]:
# SRAG Total
print("=== SRAG Total ===")
treino_srag, tl_srag, vl_srag, _ = criar_dataset_tft(df_series['casos'], janela=16)
tft_srag, tr_srag = treinar_tft(treino_srag, tl_srag, vl_srag)
print(f"Epocas: {tr_srag.current_epoch} | Val loss: {tr_srag.callback_metrics.get('val_loss'):.2f}")
y_pred_srag, y_real_srag = avaliar_tft(tft_srag, vl_srag)
torch.save(tft_srag.state_dict(), '/home/valentim/divea/models/tft_srag_total.pt')

# Influenza
print("\n=== Influenza ===")
serie_flu = df_series['influenza'].copy()
serie_flu = serie_flu[(serie_flu.index < '2020-01-01') | (serie_flu.index >= '2022-01-01')]
treino_flu, tl_flu, vl_flu, _ = criar_dataset_tft(serie_flu, janela=8)
tft_flu, tr_flu = treinar_tft(treino_flu, tl_flu, vl_flu)
print(f"Epocas: {tr_flu.current_epoch} | Val loss: {tr_flu.callback_metrics.get('val_loss'):.2f}")
y_pred_flu, y_real_flu = avaliar_tft(tft_flu, vl_flu)
torch.save(tft_flu.state_dict(), '/home/valentim/divea/models/tft_influenza.pt')

# VSR
print("\n=== VSR ===")
treino_vsr, tl_vsr, vl_vsr, _ = criar_dataset_tft(df_series['vsr'], janela=8)
tft_vsr, tr_vsr = treinar_tft(treino_vsr, tl_vsr, vl_vsr)
print(f"Epocas: {tr_vsr.current_epoch} | Val loss: {tr_vsr.callback_metrics.get('val_loss'):.2f}")
y_pred_vsr, y_real_vsr = avaliar_tft(tft_vsr, vl_vsr)
torch.save(tft_vsr.state_dict(), '/home/valentim/divea/models/tft_vsr.pt')

print("\n=== Todos os modelos salvos ===")

In [ ]:
print(f"y_pred_srag shape: {y_pred_srag.shape}")
print(f"y_real_srag shape: {y_real_srag.shape}")
print(f"Amostra real: {y_real_srag[0]}")
print(f"Amostra pred: {y_pred_srag[0, :, 3]}")


In [ ]:
# Reavaliar cada modelo separadamente
print("=== SRAG Total ===")
y_pred_srag, y_real_srag = avaliar_tft(tft_srag, vl_srag)

print("\n=== Influenza ===")
y_pred_flu, y_real_flu = avaliar_tft(tft_flu, vl_flu)

print("\n=== VSR ===")
y_pred_vsr, y_real_vsr = avaliar_tft(tft_vsr, vl_vsr)

In [ ]:
print(f"SRAG val loss: {tr_srag.callback_metrics.get('val_loss'):.2f}")
print(f"VSR val loss: {tr_vsr.callback_metrics.get('val_loss'):.2f}")
print(f"SRAG epocas: {tr_srag.current_epoch}")
print(f"VSR epocas: {tr_vsr.current_epoch}")


In [ ]:
print("=== SRAG Total ===")
treino_srag, tl_srag, vl_srag, _ = criar_dataset_tft(df_series['casos'], janela=16)
tft_srag, tr_srag = treinar_tft(treino_srag, tl_srag, vl_srag, epochs=120)
print(f"Epocas: {tr_srag.current_epoch} | Val loss: {tr_srag.callback_metrics.get('val_loss'):.2f}")
y_pred_srag, y_real_srag = avaliar_tft(tft_srag, vl_srag)
torch.save(tft_srag.state_dict(), '/home/valentim/divea/models/tft_srag_total.pt')
print("SRAG salvo.")


In [ ]:
print("=== VSR ===")
treino_vsr, tl_vsr, vl_vsr, _ = criar_dataset_tft(df_series['vsr'], janela=8)
tft_vsr, tr_vsr = treinar_tft(treino_vsr, tl_vsr, vl_vsr, epochs=120)
print(f"Epocas: {tr_vsr.current_epoch} | Val loss: {tr_vsr.callback_metrics.get('val_loss'):.2f}")
y_pred_vsr, y_real_vsr = avaliar_tft(tft_vsr, vl_vsr)
torch.save(tft_vsr.state_dict(), '/home/valentim/divea/models/tft_vsr.pt')
print("VSR salvo.")

In [ ]:
print(df_series['vsr'].describe())
print(f"\nZeros: {(df_series['vsr'] == 0).sum()}")
print(f"Valores > 0: {(df_series['vsr'] > 0).sum()}")

In [ ]:
print("=== VSR (janela 16, lr menor) ===")
treino_vsr, tl_vsr, vl_vsr, _ = criar_dataset_tft(df_series['vsr'], janela=16)

model_vsr = TemporalFusionTransformer.from_dataset(
    treino_vsr, learning_rate=0.01, hidden_size=64,
    attention_head_size=2, dropout=0.1, hidden_continuous_size=16,
    loss=QuantileLoss(), optimizer='adam', log_interval=-1,
    reduce_on_plateau_patience=6,
)
early_stop = EarlyStopping(monitor='val_loss', patience=15, mode='min')
trainer_vsr = Trainer(
    max_epochs=150, accelerator='gpu', devices=1,
    callbacks=[early_stop], enable_progress_bar=False,
    log_every_n_steps=1, logger=False,
)
trainer_vsr.fit(model_vsr, tl_vsr, vl_vsr)
print(f"Epocas: {trainer_vsr.current_epoch} | Val loss: {trainer_vsr.callback_metrics.get('val_loss'):.2f}")
y_pred_vsr, y_real_vsr = avaliar_tft(model_vsr, vl_vsr)
torch.save(model_vsr.state_dict(), '/home/valentim/divea/models/tft_vsr.pt')
print("VSR salvo.")

In [ ]:
print("=== Comparativo Final TFT vs LSTM ===")
print(f"{'Modelo':<20} {'TFT MAE+1':>10} {'TFT MAE+4':>10} {'LSTM MAE+1':>11} {'LSTM MAE+4':>11}")
print("-" * 65)
print(f"{'SRAG Total':<20} {'61.9':>10} {'67.8':>10} {'89.9':>11} {'175.8':>11}")
print(f"{'Influenza':<20} {'10.1':>10} {'12.1':>10} {'21.7':>11} {'44.9':>11}")
print(f"{'VSR':<20} {'10.3':>10} {'11.8':>10} {'19.2':>11} {'31.2':>11}")


In [3]:
import pandas as pd
import numpy as np
import torch
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

def gerar_previsao(serie, janela, modelo_path, nome):
    df = serie.reset_index()
    df.columns = ['data', 'casos']
    df['time_idx'] = range(len(df))
    df['grupo'] = 'parana'
    df['casos'] = df['casos'].astype(float)
    df['semana'] = df['data'].dt.isocalendar().week.astype(str)
    df['mes'] = df['data'].dt.month.astype(str)

    SPLIT = int(len(df) * 0.7)
    treino = TimeSeriesDataSet(
        df[df['time_idx'] <= SPLIT],
        time_idx='time_idx', target='casos', group_ids=['grupo'],
        min_encoder_length=janela, max_encoder_length=janela,
        min_prediction_length=4, max_prediction_length=4,
        time_varying_known_categoricals=['semana', 'mes'],
        time_varying_unknown_reals=['casos'],
        target_normalizer=GroupNormalizer(groups=['grupo'], transformation='softplus'),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
    )

    hidden = 64 if nome == 'vsr' else 32
    model = TemporalFusionTransformer.from_dataset(
        treino, hidden_size=hidden, attention_head_size=2,
        dropout=0.1, hidden_continuous_size=16, loss=QuantileLoss(),
    )
    model.load_state_dict(torch.load(modelo_path, map_location='cpu'))
    model.eval()

    dataset_pred = TimeSeriesDataSet.from_dataset(
        treino, df, predict=True, stop_randomization=True
    )
    loader = dataset_pred.to_dataloader(train=False, batch_size=1, num_workers=0)

    with torch.no_grad():
        for batch in loader:
            x, _ = batch
            out = model(x)
            pred = out['prediction'].cpu().numpy()[0]

    ultima_data = serie.index[-1]
    datas = [ultima_data + pd.Timedelta(weeks=i+1) for i in range(4)]

    return pd.DataFrame({
        'data': datas,
        'p10': pred[:, 0],
        'p25': pred[:, 1],
        'p50': pred[:, 3],
        'p75': pred[:, 4],
        'p90': pred[:, 5],
        'modelo': nome
    })

df_series = pd.read_parquet('/home/valentim/divea/data/processed/series_semanais.parquet')

# Gerar previsoes
resultados = []

print("SRAG...")
r = gerar_previsao(df_series['casos'], 16, '/home/valentim/divea/models/tft_srag_total.pt', 'srag')
resultados.append(r)

print("Influenza...")
serie_flu = df_series['influenza'].copy()
serie_flu = serie_flu[(serie_flu.index < '2020-01-01') | (serie_flu.index >= '2022-01-01')]
r = gerar_previsao(serie_flu, 8, '/home/valentim/divea/models/tft_influenza.pt', 'influenza')
resultados.append(r)

print("VSR...")
r = gerar_previsao(df_series['vsr'], 16, '/home/valentim/divea/models/tft_vsr.pt', 'vsr')
resultados.append(r)

df_prev = pd.concat(resultados, ignore_index=True)
df_prev.to_parquet('/home/valentim/divea/data/processed/previsoes_tft.parquet', index=False)
print("Salvo.")
print(df_prev)

SRAG...


/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/pytorch_forecasting/models/temporal_fusion_transformer/__init__.py:143: UserWarning: In pytorch-forecasting models, on versions 1.1.X, the default optimizer defaults to 'adam', if pytorch_optimizer is not installed, otherwise it defaults to 'ranger' from pytorch_optimizer. From version 1.2.0, the default optimizer will be 'adam' regardless 

Influenza...
VSR...
Salvo.
         data           p10           p25         p50         p75         p90  \
0  2026-04-27  4.841290e+02  5.455788e+02  652.929260  696.845947  743.802979   
1  2026-05-04  5.894120e+02  6.314761e+02  708.478760  745.467407  782.179382   
2  2026-05-11  6.585786e+02  6.830989e+02  732.123962  763.181641  799.045166   
3  2026-05-18  7.261652e+02  7.544823e+02  804.604919  837.650085  874.546692   
4  2026-04-27  1.913923e+01  2.033712e+01   25.608948   27.175219   28.511923   
5  2026-05-04  2.436632e+01  3.024953e+01   39.323502   43.078861   46.020390   
6  2026-05-11  3.535191e+01  3.871179e+01   52.949364   57.763611   63.646645   
7  2026-05-18  2.822958e+01  3.577058e+01   44.127708   46.914810   51.178444   
8  2026-04-27  1.923424e-21  1.039832e-29   14.262791   17.264389   19.989454   
9  2026-05-04  3.852948e-23  3.279360e-31    1.568380    5.815177    7.965302   
10 2026-05-11  3.292621e-24  2.908057e-31    2.593294    6.128601   10.039744   
1

/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/pytorch_forecasting/models/temporal_fusion_transformer/__init__.py:143: UserWarning: In pytorch-forecasting models, on versions 1.1.X, the default optimizer defaults to 'adam', if pytorch_optimizer is not installed, otherwise it defaults to 'ranger' from pytorch_optimizer. From version 1.2.0, the default optimizer will be 'adam' regardless 